In [ ]:
!pip install -q segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Paths
IMAGES_DIR = "/content/drive/MyDrive/data_processed/unet_dataset/images"
MASKS_DIR  = "/content/drive/MyDrive/data_processed/unet_dataset/masks"

# Config
NUM_CLASSES = 6
IMAGE_SIZE = 512
BATCH_SIZE = 8
MAX_EPOCHS = 40
LR = 1e-4
PATIENCE = 7
DEVICE = "cuda"

torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)

Device: cuda


In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.9,1.1),
             translate_percent=(0.05,0.05),
             rotate=(-20,20),
             p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.CLAHE(p=0.3),
    A.Normalize(mean=(0.5,0.5,0.5),
                std=(0.5,0.5,0.5)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.5,0.5,0.5),
                std=(0.5,0.5,0.5)),
    ToTensorV2(),
])

In [ ]:
class DentalDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_name = self.file_list[idx]
        base = os.path.splitext(img_name)[0]
        mask_name = base + ".png"

        image = cv2.imread(os.path.join(IMAGES_DIR, img_name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(os.path.join(MASKS_DIR, mask_name),
                          cv2.IMREAD_GRAYSCALE)

        augmented = self.transform(image=image, mask=mask)
        return augmented["image"], augmented["mask"].long()

In [ ]:
all_files = sorted([
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith((".jpg", ".jpeg"))
])

train_files, temp_files = train_test_split(
    all_files, test_size=0.30, random_state=42
)

val_files, test_files = train_test_split(
    temp_files, test_size=0.50, random_state=42
)

print("Train:", len(train_files))
print("Val:", len(val_files))
print("Test:", len(test_files))

train_dataset = DentalDataset(train_files, train_transform)
val_dataset   = DentalDataset(val_files, val_transform)
test_dataset  = DentalDataset(test_files, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=4, pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=4, pin_memory=True)

Train: 262
Val: 56
Test: 57


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
    decoder_attention_type="scse"
).to(DEVICE)

dice_loss = smp.losses.DiceLoss(mode="multiclass")
ce_loss   = nn.CrossEntropyLoss()

def criterion(outputs, targets):
    return 0.6 * dice_loss(outputs, targets) + \
           0.4 * ce_loss(outputs, targets)

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=LR,
                              weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=MAX_EPOCHS,
    eta_min=1e-6
)

scaler = torch.amp.GradScaler("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [ ]:
best_val_loss = float("inf")
early_stop_counter = 0

for epoch in range(MAX_EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_loss = 0

    for images, masks in tqdm(train_loader):
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader)

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(DEVICE)
            masks  = masks.to(DEVICE)

            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, masks)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val   Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(),
                   "/content/drive/MyDrive/best_dental_unet_512.pth")
        print("🔥 Best model saved")
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        print(f"Early stop counter: {early_stop_counter}/{PATIENCE}")

    if early_stop_counter >= PATIENCE:
        print("⛔ Early stopping triggered")
        break

  0%|          | 0/33 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100%|██████████| 33/33 [02:10<00:00,  3.96s/it]



Epoch 1/40
Train Loss: 1.1527
Val   Loss: 1.0890
🔥 Best model saved


100%|██████████| 33/33 [00:12<00:00,  2.65it/s]



Epoch 2/40
Train Loss: 0.9809
Val   Loss: 0.9415
🔥 Best model saved


100%|██████████| 33/33 [00:13<00:00,  2.38it/s]


...


Epoch 38/40
Train Loss: 0.3807
Val   Loss: 0.4898
Early stop counter: 1/7


100%|██████████| 33/33 [00:12<00:00,  2.66it/s]



Epoch 39/40
Train Loss: 0.3842
Val   Loss: 0.4889
Early stop counter: 2/7


100%|██████████| 33/33 [00:12<00:00,  2.73it/s]



Epoch 40/40
Train Loss: 0.3886
Val   Loss: 0.4899
Early stop counter: 3/7


In [ ]:
def evaluate(loader):
    model.eval()
    iou_per_class = [[] for _ in range(NUM_CLASSES)]

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(DEVICE)
            masks  = masks.to(DEVICE)

            with torch.amp.autocast("cuda"):
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            for cls in range(NUM_CLASSES):
                pred_inds = preds == cls
                target_inds = masks == cls

                intersection = (pred_inds & target_inds).float().sum()
                union = pred_inds.float().sum() + target_inds.float().sum() - intersection

                if union > 0:
                    iou = (intersection / union).item()
                    iou_per_class[cls].append(iou)

    mean_iou = np.mean([np.mean(c) for c in iou_per_class if len(c) > 0])

    return mean_iou, iou_per_class


print("\n===== FINAL TEST RESULTS =====")

model.load_state_dict(
    torch.load("/content/drive/MyDrive/best_dental_unet_512.pth")
)

test_miou, per_class_iou = evaluate(test_loader)

print("Test mIoU:", round(test_miou, 4))

for i, cls_scores in enumerate(per_class_iou):
    if len(cls_scores) > 0:
        print(f"Class {i} IoU:", round(np.mean(cls_scores), 4))


===== FINAL TEST RESULTS =====
Test mIoU: 0.5035
Class 0 IoU: 0.8103
Class 1 IoU: 0.8086
Class 2 IoU: 0.5018
Class 3 IoU: 0.0568
Class 4 IoU: 0.1762
Class 5 IoU: 0.667
